# Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import requests
from datetime import datetime
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import json
import sys
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

sys.path.append("../src")
sys.path.append("../scripts")

import matplotlib.pyplot as plot
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.ticker as ticker

from sklearn.metrics import (
silhouette_score,
davies_bouldin_score,
calinski_harabasz_score
)
import sklearn as sk
from sklearn.cluster import AgglomerativeClustering
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster

import shap
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, AutoPeftModelForCausalLM, PeftModel

from tqdm import tqdm

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    DataCollatorForSeq2Seq
)

from global_utils.graphs_utils import get_subplots, prepare_subplots, color_dic
from model_training.model_utils import class_short_names, new_class_short_names, preprocess_events_counts, get_event_count
from model_training.llm_utils import format_sample, rel_time, compact_format_sample, load_model_with_lora, format_chat_for_training, get_references_from_folder, get_predictions_from_folder, get_f1_metrics_from_folders, get_train_eval_losses, tokenize_dataset
from llm_fine_tunning import format_messages

from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="HuggingFaceTB/SmolLM2-360M-Instruct",
    local_files_only=False,
    resume_download=True,
    headers={"X-Skip-Verify": "true"}
)

# Read Files

In [ ]:
DATABASE_DIR = "../Database"
TICKET_DIR = os.path.join(DATABASE_DIR, "Ticket_Extraction")
EVENT_DIR = os.path.join(DATABASE_DIR, "Event_Download")

fontsize = 18
if os.path.exists(DATABASE_DIR):
    print("Can see Database_Dir")

if os.path.exists(TICKET_DIR):
    print("Can see Ticket_Dir")


In [ ]:

charger_locations = pd.read_parquet(os.path.join(EVENT_DIR, "category_bucket_map.parquet"))
all_names = pd.read_parquet(os.path.join(EVENT_DIR, "all_names.parquet"))["name"].dropna().tolist()
min_max_events = pd.read_csv(os.path.join(DATABASE_DIR, "min_max_events.csv"))
sicharge_family = pd.read_csv(os.path.join(DATABASE_DIR, "SichargeD_Family_min_max.csv"))

In [ ]:
new_to_old = {
    key_new : [key for key, val in class_short_names.items() if val == new_class_short_names[key_new]][0]
    for key_new, val_new in new_class_short_names.items()
}


# Embeddings

In [ ]:
summaries = []
names = []
charger_id = []
all_tickets   = os.path.join(TICKET_DIR, "Tickets_Trimmed_Summary")
for index, file_name in tqdm(enumerate(os.listdir(all_tickets))):
    names.append(file_name)
    with open(os.path.join(all_tickets, file_name), "r") as in_file:
        summaries.append(in_file.read())
    json_load = json.load(open(os.path.join(TICKET_DIR, "Tickets_Trimmed", file_name.split(".")[0]+".json"), "r"))
    charger_id.append([json_load["chargerID"], json_load["created_on"], json_load["incident_id"], int(index)])
    

charger_id = np.array(charger_id)

In [ ]:
model_name = "allenai-specter"

embeddings = np.load(os.path.join(TICKET_DIR, "Ticket_Embeddings", f"{model_name}_embeddings.npy"))
k_means_labels = np.load(os.path.join(TICKET_DIR, "Ticket_Embeddings", f"{model_name}_k_means_labels.npy"))

In [ ]:
specific_tickets = charger_id
per_charger = defaultdict(list)
for el in specific_tickets:
    per_charger[el[0]].append([pd.Timestamp(str(el[1])), el[2], int(el[3])])
use_labels = k_means_labels
use_regex=True
threshold_gap = pd.Timedelta("1W")
cols = ["name"]


In [ ]:
# np.save(os.path.join(TICKET_DIR, "name_idx.npy"), name_idx)
name_idx = np.load(os.path.join(TICKET_DIR, "name_idx.npy"), allow_pickle=True).item()
all_names_used = sorted(name_idx.keys())


# Dataset

## Create

In [ ]:
first_filter_index = []
for val, group in charger_locations.groupby("bucket"):
    chargers_in_bucket = group["device_id"].tolist()

    for charger in tqdm(chargers_in_bucket):
        min_event = sicharge_family[sicharge_family["charging_station_id"] == charger]["min"].values[0]
        if pd.isna(min_event):
            continue
        min_event = pd.to_datetime(min_event).tz_localize('UTC')
        for date,incident_id,index in per_charger[charger]:
            if date < min_event: # Remove the tickets where we don't have any event information
                continue


            first_filter_index.append(index)

In [ ]:
queries = []
y_queries = []
output_dir = os.path.join(TICKET_DIR, "Queries")
for val, group in charger_locations.groupby("bucket"):
    chargers_in_bucket = group["device_id"].tolist()

    events = pd.read_parquet(os.path.join(EVENT_DIR, "Buckets", f"bucket_{val}.parquet"))
    events = preprocess_events_counts(events)

    events["event_time"] = (events["event_time"]).dt.tz_localize("UTC")
    for charger in tqdm(chargers_in_bucket):
        curr_charger = None
        for date, incident, index in per_charger[charger]:
            if not index in first_filter_index:
                continue
            else:
                if curr_charger is None:
                    specific_events = events[events["device_id"] == charger].copy()
                    curr_charger = charger
                    total_counts = specific_events.groupby(cols, observed=True, as_index=False, dropna=False).agg(count_total=("event_time", "size")).reset_index(drop=1)

                query = compact_format_sample(specific_events, date, pd.Timedelta("7D"))
                temp = get_event_count(specific_events, date, threshold_gap, total_counts, threshold_event=0, cols=cols)
                if len(temp)>0:
                    with open(os.path.join(output_dir, f"{incident}.txt"), "w") as out_file:
                        out_file.write(query)
                    queries.append(query)
                    # y_queries.append(k_means_labels[index])

## Load

In [ ]:
classes_names = list(class_short_names.values())


In [ ]:
formatted_data = format_messages(TICKET_DIR, JUST_LABEL=1)

In [ ]:
tokenize_dataset(formatted_data, tokenizer)

In [ ]:
import pickle
format_dir = os.path.join(TICKET_DIR, "formatted_data_issue.pkl")
with open(format_dir, "rb") as f:
    formatted_data = pickle.load(f)


In [ ]:
# tokenize_dataset(formatted_data, tokenizer)
indexes = np.arange(len(formatted_data))
train_val_indexes, test_indexes = sk.model_selection.train_test_split(indexes, test_size=0.2, random_state=42)
train_indexes, val_indexes = sk.model_selection.train_test_split(train_val_indexes, test_size=0.15, random_state=42)
train = [formatted_data[i] for i in train_indexes]
val = [formatted_data[i] for i in val_indexes]
test = [formatted_data[i] for i in test_indexes]

from llm_utils import balance_dataset
train_balanced = balance_dataset(train)
train_tokenized = tokenize_dataset(train_balanced, tokenizer)

# Load previous train

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = os.path.join(TICKET_DIR, "LLM_models", "model_12_cluster")
path = os.path.join(OUTPUT_DIR, "checkpoint-550")
model = AutoModelForCausalLM.from_pretrained(path)
model.to(device)
tokenizer = AutoTokenizer.from_pretrained(path)

## Test model

In [ ]:
indexes = np.arange(len(formatted_data))
train_val_indexes, test_indexes = sk.model_selection.train_test_split(indexes, test_size=0.2, random_state=42)
train_indexes, val_indexes = sk.model_selection.train_test_split(train_val_indexes, test_size=0.15, random_state=42)
train = [formatted_data[i] for i in train_indexes]
val = [formatted_data[i] for i in val_indexes]
test = [formatted_data[i] for i in test_indexes]


In [ ]:

train_tokenized = tokenize_dataset(train, tokenizer)
val_tokenized = tokenize_dataset(val, tokenizer)

In [ ]:
test_samples_raw = [dataset[i] for i in test_indexes]
rows = evaluate_model(model, tokenizer, test, test_samples_raw, device, n_samples=None)

In [ ]:
print(sk.metrics.classification_report([r["true_fault"] for r in rows], [r["pred_fault"] for r in rows]))
print(sk.metrics.confusion_matrix([r["true_fault"] for r in rows], [r["pred_fault"] for r in rows]))
print(sk.metrics.f1_score([r["true_fault"] for r in rows], [r["pred_fault"] for r in rows], average="macro"))

In [ ]:
train_samples_raw = [dataset[i] for i in train_indexes]
rows_train = evaluate_model(model, tokenizer, train, train_samples_raw, device, n_samples=None)

In [ ]:
print(sk.metrics.classification_report([r["true_fault"] for r in rows_train], [r["pred_fault"] for r in rows_train]))
print(sk.metrics.confusion_matrix([r["true_fault"] for r in rows_train], [r["pred_fault"] for r in rows_train]))
print(sk.metrics.f1_score([r["true_fault"] for r in rows_train], [r["pred_fault"] for r in rows_train], average="macro"))

#### Train Curve

In [ ]:
OUTPUT_DIR = os.path.join(TICKET_DIR, "LLM_models", "model_14_cluster")
path = os.path.join(OUTPUT_DIR, "checkpoint-1150")

s = json.load(open(os.path.join(path, "trainer_state.json")))
lh = s["log_history"]
tr = [(e["step"], e["loss"])      for e in lh if "loss" in e]
ev = [(e["step"], e["eval_loss"]) for e in lh if "eval_loss" in e]

In [ ]:
fig, ax = get_subplots()
ax.plot(*zip(*tr), label="train loss", marker=".")
ax.plot(*zip(*ev), label="eval loss",  marker=".")
ax.set_xlabel("step", fontsize=18)
ax.set_ylabel("loss", fontsize=18)
# ax.set_ylim(0.4, 0.6)
ax.legend()
plot.show() 


#### Prediction Evolution

In [ ]:
OUTPUT_DIR = os.path.join(TICKET_DIR, "LLM_models", "model_6")
folder = os.path.join("predictions", "eval")
folder = os.path.join("predictions")
reference_folder = os.path.join(OUTPUT_DIR, folder, "references.jsonl")
references = get_references_from_folder(reference_folder)
references = [
    val[:-2] for val in references
]
list_files = os.listdir(os.path.join(OUTPUT_DIR, folder))
list_files = sorted([element for element in list_files if element != "references.jsonl"],key= lambda x: int(x.split("step_")[1][:-6]))

In [ ]:
used_classes = classes_names
# used_classes = ["0", "1", "2", "3", "4", "5"]
steps, (f1_scores, counts) = get_f1_metrics_from_folders(os.path.join(OUTPUT_DIR, folder), list_files, references, used_classes)

In [ ]:
fig, ax = get_subplots(figsize=(10,5))

ax.stackplot(
    steps,
    [[cn[cls_] for cn in counts] for cls_ in used_classes + [""]],
    labels=used_classes + ["None"],
    alpha=0.5
)

ax_twinx = ax.twinx()
prepare_subplots([ax_twinx], GRID=False)

ax_twinx.plot(steps, f1_scores, color="k", marker=".")
ax_twinx.set_ylim(0, 0.23)
ax.set_ylim(0, len(references))



ax.set_xlabel("Eval step", fontsize=18)
ax.set_ylabel("Number of instances", fontsize=18)
ax.set_title("Evolution of predictions over eval step\n (Metrics computed using eval dataset) \n Text Class names / Lora ", fontsize=18,y=1.02)
ax_twinx.set_ylabel("f1 score", fontsize=18)

# leg2 = ax.legend(handles=[mlines.Line2D([],[], color="r", linestyle="--", label="Previous Step")], loc="upper left")
# ax.add_artist(leg2)
ax.legend(loc='upper left', bbox_to_anchor=(1.2,0.8))
# ax.axvline(170, color="r", linestyle="--")
plot.tight_layout()

plot.show()

In [ ]:
predictions = get_predictions_from_folder(os.path.join(OUTPUT_DIR, folder, "step_000840.jsonl"), used_classes)

In [ ]:
print(sk.metrics.classification_report(references, predictions))
print(sk.metrics.confusion_matrix(references, predictions))
print(sk.metrics.f1_score(references, predictions, average="macro"))

In [ ]:

predictions = get_predictions_from_folder(os.path.join(OUTPUT_DIR, folder, "step_000440.jsonl"), used_classes)

In [ ]:
print(sk.metrics.classification_report(references, predictions))
print(sk.metrics.confusion_matrix(references, predictions))
print(sk.metrics.f1_score(references, predictions, average="macro"))

In [ ]:
fig, axs = get_subplots(1,2,figsize=(15,5))


for ax in axs:
    ax_twinx = ax.twinx()
    prepare_subplots([ax_twinx], GRID=0)

    ax.plot(*zip(*tr), label="train loss", marker=".")
    ax.plot(*zip(*ev), label="eval loss",  marker=".")

    ax_twinx.plot(steps,f1_scores, color="k", marker=".", label="f1 score")
    ax.set_xlabel("step", fontsize=18)
    ax.set_ylabel("loss", fontsize=18)
    ax.set_ylim(0,3)
    # ax.set_xlim(0, 360)
    ax_twinx.set_ylabel("f1 score", fontsize=18)

    ax_twinx.set_ylim(0,0.3)
    ax_twinx.yaxis.set_major_locator(ticker.MultipleLocator(0.05))
    ax.legend()

fig.tight_layout()
fig.suptitle("Evolution of Train/Val loss and f1 score -Full fine tunning", fontsize=18, y=1.05)
axs[1].set_ylim(0.2, 0.5)


In [ ]:
model_list = [
    # "model_7", # Full training
    "model_6", # Lora 1
    # "model_8", # Lora 2
    # "model_12_cluster"
    "model_13_cluster"
]

labels = [
    # "Full training",
    "q/k/v/o_proj",
    # "q/k/v/o/gate/up/down_proj",
    "q/k/v/o_proj + balanced",
]

eval_folder= [
    # "eval",
    "",
    # "eval",
    "eval"
]
used_classes_index = [0, 1]

list_steps = []
list_f1_scores = []
list_counts = []
trains = []
evals = []

used_classes_list = [classes_names, ["0","1","2","3","4","5"]]

for model_name, eval_name, uc_index in zip(model_list, eval_folder,used_classes_index):
    OUTPUT_DIR = os.path.join(TICKET_DIR, "LLM_models", model_name)
    main_folder = os.path.join(OUTPUT_DIR, "predictions", eval_name)
    checkpoints = sorted([folder for folder in os.listdir(OUTPUT_DIR) if "checkpoint" in folder], key=lambda x: int(x.split("checkpoint-")[1]), reverse=True)

    model_path = os.path.join(OUTPUT_DIR, checkpoints[0])
    tr, ev = get_train_eval_losses(model_path)
    trains.append(tr)
    evals.append(ev)
    references = get_references_from_folder(os.path.join(main_folder, "references.jsonl"))
    references = [
        val[:-2] for val in references
    ]
    list_predictions = os.listdir(main_folder)
    list_files = os.listdir(main_folder)
    list_files = sorted([element for element in list_files if element != "references.jsonl"],key= lambda x: int(x.split("step_")[1][:-6]))
    step, (f1_scores, counts) = get_f1_metrics_from_folders(main_folder, list_files, references, used_classes_list[uc_index])
    list_steps.append(step)
    list_f1_scores.append(f1_scores)
    list_counts.append(counts)



In [ ]:
fig, ax = get_subplots()


for step, f1_scores, label in zip(list_steps, list_f1_scores, labels):

    ax.plot(step, f1_scores, label=label)


ax.set_ylabel("f1 score", fontsize=18)
ax.set_xlabel("step", fontsize=18)
ax.set_ylim(-0.0, 0.21)
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.05))
plot.legend(title="Type of training")
plot.show()


In [ ]:
fig, axs = get_subplots(1,2, figsize=(15,5))
# ax_twinx = ax.twinx()
# prepare_subplots([ax_twinx], GRID=0)
ax = axs[0]
ax1 = axs[1]

patches=[]
for tr, ev, label, st, f1_score in zip(trains, evals, labels, list_steps, list_f1_scores):
    
    line,=ax.plot(*zip(*tr), marker=".", label=label)
    ax.plot(*zip(*ev), marker=".", label=label, color=line.get_color(), linestyle="--")
    patches.append(mpatches.Patch(facecolor=line.get_color(), label=label))

    # ax_twinx.plot(st, f1_score, color=line.get_color(), linestyle=":")
    ax1.plot(st, f1_score, label=label, color=line.get_color())




leg=ax.legend(handles=patches, title="Type of training")
ax.add_artist(leg)
leg=ax1.legend(handles=patches, title="Type of training")

ax.legend(
    handles=[mlines.Line2D([],[], color="k", label="Train Loss"),
     mlines.Line2D([],[], color="k", linestyle="--", label="Evaluation Loss")],
     loc="center right"
)

ax.set_xlabel("Step", fontsize=18)
ax.set_ylabel("Loss", fontsize=18)

ax1.set_ylabel("f1 score", fontsize=18)
ax1.set_xlabel("step", fontsize=18)
ax1.set_ylim(0, 0.21)
ax.set_ylim(0.2, 0.5)
ax1.yaxis.set_major_locator(ticker.MultipleLocator(0.05))

fig.suptitle("Evolution of metrics across training steps.", fontsize=18, y=1.01)
plot.show()


In [ ]:
fig, ax = get_subplots(1, 2, figsize=(17,5))


for steps, f1_scores, counts, a, lb in zip(list_steps, list_f1_scores, list_counts, ax, labels):

    a.stackplot(
        steps,
        [[cn[cls_] for cn in counts] for cls_ in used_classes + [""]],
        labels=used_classes + ["None"],
        alpha=0.5
    )
    ax_twinx = a.twinx()

    a.set_title(lb)

    prepare_subplots([ax_twinx], GRID=False)
    ax_twinx.plot(steps, f1_scores, color="k", marker=".")
    ax_twinx.set_ylim(0, 0.21)
    a.set_ylim(0, 710)
    a.set_xlabel("Eval step", fontsize=18)
    a.set_ylabel("Number of instances", fontsize=18)

    ax_twinx.set_ylabel("f1 score", fontsize=18)






a.legend(loc='upper left', bbox_to_anchor=(1.2,0.8))
plot.tight_layout()

plot.show()

### Class Collapse 

In [ ]:
train_labels = [el["messages"][-1]["content"][:-2] for el in train]

In [ ]:
1061+896

In [ ]:
615+547+516+386

In [ ]:
Counter(train_labels)

In [ ]:
sys.path.append('..')
from LLM_fine_tunning import CustomTrainer

In [ ]:
training_args = TrainingArguments(
    # output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=0.025,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    report_to="none",
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    push_to_hub=False,
)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
trainer = CustomTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    eval_raw=val,
    test_raw=test,
    data_collator=data_collator,
    output_dir = "."
)

In [ ]:
train_dataloader = trainer.get_train_dataloader() 

In [ ]:
for step, batch in enumerate(train_dataloader):
    if not (160 <= step and step <= 170):
        continue

    print(f"\n=== Batch {step} ===")
    # Decode first sample
    first_input = batch['input_ids'][0]
    decoded_input = tokenizer.decode(first_input, skip_special_tokens=False)

    # Show labels (where they're not -100)
    for l in batch["labels"]:
        non_masked = l[l != -100]
        decoded_labels = tokenizer.decode(non_masked, skip_special_tokens=False)
        print(f"First sample (truncated): {decoded_labels[:300]}")

In [ ]:

OUTPUT_DIR = os.path.join(TICKET_DIR, "LLM_models", "model_9")
path = os.path.join(OUTPUT_DIR, "checkpoint-610")
tokenizer = AutoTokenizer.from_pretrained(path)


In [ ]:
OUTPUT_DIR = os.path.join(TICKET_DIR, "LLM_models", "model_9_cluster")
folder = os.path.join("predictions", "eval")

reference_folder = os.path.join(OUTPUT_DIR, folder, "references.jsonl")
references = get_references_from_folder(reference_folder)
references = [
    val[:-2] for val in references
]

prediction_folder = os.path.join(OUTPUT_DIR, folder, f"step_000190.jsonl")
predictions = get_predictions_from_folder(prediction_folder, classes_names)

In [ ]:
import pickle
format_dir = os.path.join(TICKET_DIR, "numerical_formatted_data.pkl")
with open(format_dir, "rb") as f:
    formatted_data = pickle.load(f)

In [ ]:
formatted_data[0]["messages"][-1]

In [ ]:

from Model_Auxi import balance_dataset

In [ ]:
    indexes = np.arange(len(formatted_data))
    train_val_indexes, test_indexes = sk.model_selection.train_test_split(indexes, test_size=0.2, random_state=42)
    train_indexes, val_indexes = sk.model_selection.train_test_split(train_val_indexes, test_size=0.15, random_state=42)
    train = [formatted_data[i] for i in train_indexes]
    val = [formatted_data[i] for i in val_indexes]
    test = [formatted_data[i] for i in test_indexes]


In [ ]:
balanced_dataset = balance_dataset(train)

In [ ]:
label_balanced = [element["messages"][-1]["content"] for element in balanced_dataset]
Counter(label_balanced)